# Instalación y Librerías

In [ ]:
# pip install ollama

In [2]:
# Para tratamiento de datos
import pandas as pd
import numpy as np
import re #para llamar a Expresiones Regulares y estandarizar el nombre de las columnas.
import time
import requests

# Para visualización de datos
import matplotlib.pyplot as plt
import seaborn as sns

# Para poder visualizar todas las columnas de los DataFrames
pd.set_option('display.max_columns', None) 

# Trabajar con el sistema operativo y variables de entorno
import os 
from dotenv import load_dotenv
load_dotenv() #carga las variables del entorno .env; devuelve un true o false
import ollama
MODELO = 'gemma3:4b' 

# Conexión con MySQL
import mysql.connector
from mysql.connector import Error

# Gestión de los warnings
import warnings
warnings.filterwarnings("ignore")



In [3]:
# Cargamos las variables de entorno del archivo .env
load_dotenv()

# 1. Recuperamos la Key y el ID base
steam_key = os.getenv("steam_key")

# 2. Creamos la lista de los 21 IDs dinámicamente
lista_ids = []
for i in range(21):
    variable_name = f"steam_id_{i}"
    valor_id = os.getenv(variable_name)
    if valor_id:
        lista_ids.append(valor_id)

print(f"Se han cargado {len(lista_ids)} IDs desde el archivo .env")

Se han cargado 21 IDs desde el archivo .env


# DS Jugadores

In [ ]:
load_dotenv()
steam_key = os.getenv("steam_key")

# Cargamos los 21 IDs desde el .env
ids_proyecto = [os.getenv(f"steam_id_{i}") for i in range(21)]
ids_proyecto = [i for i in ids_proyecto if i] # Limpiamos valores None

registros_juegos = []

print(f"--- Iniciando construcción del Dataset ({len(ids_proyecto)} usuarios) ---")

for s_id in ids_proyecto:
    # Usamos HTTPS y especificamos el formato JSON explícitamente
    url = "https://api.steampowered.com/IPlayerService/GetOwnedGames/v0001/"
    params = {
        'key': steam_key,
        'steamid': s_id,
        'format': 'json',
        'include_appinfo': 1,
        'include_played_free_games': 1
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            # Accedemos a la estructura de la respuesta
            juegos = data.get('response', {}).get('games', [])
            
            for juego in juegos:
                juego['user_id'] = s_id
                registros_juegos.append(juego)
            
            print(f"✅ ID {s_id}: {len(juegos)} juegos añadidos.")
        else:
            print(f"❌ ID {s_id}: Error HTTP {response.status_code} (Posible perfil privado)")
            
    except Exception as e:
        print(f"⚠️ ID {s_id}: Error de conexión o formato: {e}")
    
    time.sleep(0.5) # Pausa para evitar el baneo de la API

# Creación del DataFrame y guardado
if registros_juegos:
    df_steam = pd.DataFrame(registros_juegos)
    df_steam.to_csv("./datasets/dataset_steam.csv", index=False)
    print(f"\n¡ÉXITO! Dataset guardado con {len(df_steam)} filas.")
else:
    print("\nNo se pudo extraer información. Revisa tu steam_key.")

def obtener_estadisticas_logros(api_key, steam_id, app_id):
    # Endpoint para logros de usuario
    url = "https://api.steampowered.com/ISteamUserStats/GetPlayerAchievements/v0001/"
    params = {'key': api_key, 'steamid': steam_id, 'appid': app_id}
    
    try:
        r = requests.get(url, params=params, timeout=5)
        if r.status_code == 200:
            data = r.json()
            achievements = data.get('playerstats', {}).get('achievements', [])
            total = len(achievements)
            ganados = sum(1 for a in achievements if a.get('achieved') == 1)
            return ganados, total
    except:
        return 0, 0
    return 0, 0

# 3. Procesar una muestra significativa (ej. los 100 registros más jugados)
# Usamos 'user_id' que es como se guardó en tu CSV anterior
df_sample = df_steam.sort_values(by='playtime_forever', ascending=False).head(100).copy()

logros_data = []
print("Enriqueciendo datos con logros... (esto tardará un poco)")

for idx, row in df_sample.iterrows():
    # CAMBIO CLAVE: Usamos 'user_id' para evitar el KeyError de tu imagen
    ganados, totales = obtener_estadisticas_logros(steam_key, row['user_id'], row['appid'])
    logros_data.append({
        'achieved': ganados,
        'total_achievements': totales,
        'completion_percentage': (ganados / totales * 100) if totales > 0 else 0
    })
    time.sleep(0.2) # Pausa de cortesía para la API

# 4. Unir y guardar el dataset final
df_extra = pd.DataFrame(logros_data)
df_final = pd.concat([df_sample.reset_index(drop=True), df_extra], axis=1)

# No necesitamos la columna de imágenes URL ya que no aporta valor, por lo que lo borramos. Tampoco necesitamos "leaderboards".
df_final = df_final.drop(columns=['img_icon_url'], errors='ignore')
df_final = df_final.drop(columns=['has_leaderboards'], errors='ignore')

#Creamos una columna nueva que indique si el juego tiene o no contenido para adultos
df_final['adult_descriptorids'] = df_final['content_descriptorids'].notna()

df_final.to_csv("./datasets/dataset_steam.csv", index=False)
print("¡Hecho! Dataset actualizado.")


--- Iniciando construcción del Dataset (21 usuarios) ---
✅ ID 76561198056243081: 41 juegos añadidos.
✅ ID 76561198842603734: 28623 juegos añadidos.
✅ ID 76561198023414915: 4646 juegos añadidos.
✅ ID 76561199080934614: 23328 juegos añadidos.
✅ ID 76561197984432884: 3043 juegos añadidos.
✅ ID 76561198254085126: 67 juegos añadidos.
✅ ID 76561198048165534: 15680 juegos añadidos.
✅ ID 76561198023455525: 8895 juegos añadidos.
✅ ID 76561198092430664: 0 juegos añadidos.
✅ ID 76561198294650349: 1345 juegos añadidos.
✅ ID 76561198203118756: 0 juegos añadidos.
✅ ID 76561197986603983: 14690 juegos añadidos.
✅ ID 76561199219841553: 0 juegos añadidos.
✅ ID 76561198212206651: 2817 juegos añadidos.
✅ ID 76561198062673538: 1272 juegos añadidos.
✅ ID 76561198306626714: 0 juegos añadidos.
✅ ID 76561198014898339: 3581 juegos añadidos.
✅ ID 76561198067053149: 3336 juegos añadidos.
✅ ID 76561199499421434: 4251 juegos añadidos.
✅ ID 76561198046160451: 1315 juegos añadidos.
✅ ID 76561198108581917: 0 juegos añ

In [15]:
df_final.head()

,appid,name,playtime_forever,img_icon_url,has_community_visible_stats,playtime_windows_forever,playtime_mac_forever,playtime_linux_forever,playtime_deck_forever,rtime_last_played,content_descriptorids,playtime_disconnected,user_id,has_leaderboards,playtime_2weeks,achieved,total_achievements,completion_percentage
0,366240,GAROU: MARK OF THE WOLVES,1073865,0940266c09476bb48f160be222e370ebed8c5315,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,76561198048165534,True,4806.0,25,28,89.285714
1,730,Counter-Strike 2,749353,8dbc71957312bbd3baea65848b545be9eae2a355,True,NaN,NaN,NaN,NaN,NaN,"[2, 5]",NaN,76561199080934614,NaN,NaN,1,1,100.000000
2,1267910,Melvor Idle,621639,88eca64c662c128de095b4e452881398ac71969c,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,76561198014898339,NaN,NaN,86,86,100.000000
3,730,Counter-Strike 2,575785,8dbc71957312bbd3baea65848b545be9eae2a355,True,NaN,NaN,NaN,NaN,NaN,"[2, 5]",NaN,76561197984432884,NaN,1510.0,1,1,100.000000
4,1085660,Destiny 2,533973,fce050d63f0a2f8eb51b603c7f30bfca2a301549,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,76561197986603983,NaN,NaN,23,23,100.000000


In [13]:
data_mapeo_contenido = {
    'content_id': [1, 2, 3, 4, 5],
    'description_es': [
        'Contenido sexual explícito o desnudez',
        'Violencia frecuente o intensa',
        'Lenguaje fuerte',
        'Temas adultos/sensibles',
        'Escenas de sangre y desmembramiento'
    ]
}

# 2. Creamos el DataFrame de mapeo
df_mapeo = pd.DataFrame(data_mapeo_contenido)

# 3. Extraemos los IDs únicos de la columna con listas
# Usamos explode para separar las listas en filas individuales y luego unique()
# Filtrando primero los nulos para evitar errores.
unique_ids = pd.Series(
    [item for sublist in df_final['content_descriptorids'].dropna() for item in sublist]
).unique()

# 4. Filtramos el df_mapeo para tener solo los que aparecen en df_final
df_descriptors_unicos = df_mapeo[df_mapeo['content_id'].isin(unique_ids)].reset_index(drop=True)
df_descriptors_unicos.to_csv("./datasets/mapeo_contenido.csv", index=False)


# DS Maestro de juegos

In [10]:
# 1. Preparación de los AppIDs únicos
appids_unicos = df_final["appid"].unique()

# --- FUNCIONES DE APOYO ---

def obtener_info_juego(appid):
    """Consulta la API de Steam para obtener detalles básicos."""
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}&l=spanish"
    try:
        response = requests.get(url)
        data = response.json()
        if data and data[str(appid)]['success']:
            details = data[str(appid)]['data']
            is_free = details.get('is_free', False)
            price = 0 if is_free else (details.get('price_overview', {}).get('final', 0) / 100)
            devs_list = details.get('developers', [])
            
            return {
                'appid': appid,
                'name': details.get('name', 'N/A'),
                'developer': devs_list[0] if devs_list else "Desconocido",
                'price': price,
                'is_free': is_free,
                'metacritic_score': details.get('metacritic', {}).get('score', 0),
                'total_recommendations': details.get('recommendations', {}).get('total', 0)
            }
    except Exception as e:
        print(f"Error en API Steam para {appid}: {e}")
    return None

def obtener_pais_empresa(empresa):
    """Obtiene el país de la empresa usando Ollama."""
    if pd.isna(empresa) or empresa == '' or empresa == 'Desconocido':
        return 'Desconocido'
    prompt = f""""
    Eres un buscador de sedes corporativas.
    Tarea: Indica el país donde se encuentra la sede principal de la empresa: '{empresa}'.
    
    Reglas críticas de respuesta:
    1. Responde ÚNICAMENTE con el nombre del país en español.
    2. NO escribas frases como "La sede está en..." o "El país es...".
    3. NO uses puntos finales ni signos de puntuación.
    4. NO des ninguna información adicional, ni ciudades, ni fechas. Lo que nos interesa es dónde están situados los HQ de la empresa.
    5. Si no conoces el país, responde exclusivamente con la palabra: Desconocido.
    """
    try:
        response = ollama.chat(model=MODELO, messages=[{'role': 'user', 'content': prompt}])
        return response['message']['content'].strip()
    except Exception:
        return "Error"

def obtener_genero_limpio(juego, lista_generos_conocidos):
    """Clasifica el género del juego usando Ollama."""
    if pd.isna(juego) or juego == '':
        return 'Desconocido'
    generos_str = ", ".join(list(set(lista_generos_conocidos)))
    prompt = f"""
    Eres un experto clasificador de videojuegos.
    Objetivo: Define el género principal del juego '{juego}'.
    Reglas estrictas:
    1. Responde con exactamente UNA palabra.
    2. El género debe ser genérico (ej. "Acción", "RPG", "Estrategia"), NO específico.
    3. Usa e spañol o inglés, el que sea más común paraese género.
    4. NO uses puntos finales ni explicaciones.
    5. IMPORTANTE: Consulta esta lista: [{generos_str}]. Si encaja, usa esa palabra exacta.
    """
    try:
        response = ollama.chat(model=MODELO, messages=[{'role': 'user', 'content': prompt}])
        return response['message']['content'].strip().split('\n')[0].replace(".", "")
    except Exception:
        return "Error"

# --- EJECUCIÓN DEL PROCESO ---

# PASO 1: Extracción de Steam (Bucle con Rate Limit)
lista_datos = []
print(f"Extrayendo datos de Steam ({len(appids_unicos)} juegos)...")
for appid in appids_unicos:
    info = obtener_info_juego(appid)
    if info:
        lista_datos.append(info)
    time.sleep(1.5) 

df_juegos_maestro = pd.DataFrame(lista_datos)

# PASO 2: Procesamiento de Países (Fuera del bucle de Steam)
print("Procesando localizaciones con Ollama...")
df_juegos_maestro['country'] = df_juegos_maestro['developer'].apply(obtener_pais_empresa)

# PASO 3: Procesamiento de Géneros (Iterativo para mantener memoria de géneros)
print("Procesando géneros con Ollama...")
generos_ya_procesados = []
df_juegos_maestro['genre'] = ''

for index, row in df_juegos_maestro.iterrows():
    genero = obtener_genero_limpio(row['name'], generos_ya_procesados)
    df_juegos_maestro.at[index, 'genre'] = genero
    if genero not in generos_ya_procesados and genero not in ['Error', 'Desconocido']:
        generos_ya_procesados.append(genero)

# PASO 4: Guardado y Visualización
df_juegos_maestro.to_csv("./datasets/juegos_maestro.csv", index=False)
print("\nProcesamiento completo. Dataset final:")
df_juegos_maestro.head()

Extrayendo datos de Steam (90 juegos)...
Procesando localizaciones con Ollama...
Procesando géneros con Ollama...

Procesamiento completo. Dataset final:


,appid,name,developer,price,is_free,metacritic_score,total_recommendations,country,genre
0,366240,GAROU: MARK OF THE WOLVES,SNK CORPORATION,9.99,False,0,897,Japón,RPG
1,730,Counter-Strike 2,Valve,0.00,True,0,4954213,Estados Unidos,Acción
2,1267910,Melvor Idle,Games by Malcs,9.75,False,0,14104,Reino Unido,RPG
3,1085660,Destiny 2,Bungie,0.00,True,83,133584,Estados Unidos,Acción
4,312660,Sniper Elite 4,Rebellion,59.99,False,78,53112,Reino Unido,Acción
